# 🤖 Omni-Data Analyst Agent
> **Brain**: Claude (Tool Use / Function Calling) · **Bridge**: MCP-style tool registry · **Frontend**: Streamlit + ngrok

### Architecture at a glance
```
User NL Prompt
      │
      ▼
  Claude API  ──tool_call──▶  Tool Registry (MCP-style)
      │                          ├─ execute_sql()   ← SQLite
      │                          ├─ list_tables()   ← Schema browser
      │                          ├─ plot_chart()    ← Matplotlib
      │                          └─ forecast()      ← Scikit-learn
      │
      ▼
  Streamlit UI (tunnelled via ngrok)
```

### Steps
1. **Cell 1** – Install deps  
2. **Cell 2** – Seed a sample SQLite database (swap with your own later)  
3. **Cell 3** – Define MCP-style tool registry  
4. **Cell 4** – Claude agent loop  
5. **Cell 5** – Streamlit app  
6. **Cell 6** – Launch with ngrok  

---
## Cell 1 · Install Dependencies

In [ ]:
# ── Install everything we need ──────────────────────────────────────────────
!pip install -q anthropic streamlit pyngrok scikit-learn matplotlib pandas

# Verify
import anthropic, streamlit, sklearn, matplotlib, pandas
print('✅ All packages installed')

---
## Cell 2 · Seed Sample SQLite Database
> Simulates a SaaS company's data warehouse. Replace with your own DB path later.

In [ ]:
import sqlite3, pandas as pd, numpy as np, random
from datetime import datetime, timedelta

DB_PATH = '/content/company_data.db'
conn = sqlite3.connect(DB_PATH)

# ── Generate realistic fake data ────────────────────────────────────────────
np.random.seed(42)
random.seed(42)

# 1. Monthly revenue (24 months)
dates = [datetime(2023,1,1) + timedelta(days=30*i) for i in range(24)]
revenue = pd.DataFrame({
    'month': [d.strftime('%Y-%m') for d in dates],
    'revenue': np.cumsum(np.random.randint(8000, 15000, 24)) + 50000,
    'new_customers': np.random.randint(20, 80, 24),
    'churned_customers': np.random.randint(5, 20, 24),
    'region': random.choices(['North America','Europe','Asia','LATAM'], k=24)
})
revenue.to_sql('monthly_revenue', conn, if_exists='replace', index=False)

# 2. Products
products = pd.DataFrame({
    'product_id': range(1, 11),
    'name': [f'Product {chr(64+i)}' for i in range(1,11)],
    'category': random.choices(['SaaS','Hardware','Consulting','Support'], k=10),
    'unit_price': np.random.randint(29, 999, 10),
    'units_sold': np.random.randint(100, 5000, 10)
})
products.to_sql('products', conn, if_exists='replace', index=False)

# 3. Customer events
events = pd.DataFrame({
    'event_id': range(1, 201),
    'customer_id': np.random.randint(1, 50, 200),
    'event_type': random.choices(['signup','purchase','upgrade','churn','support_ticket'], k=200),
    'event_date': [(datetime(2024,1,1) + timedelta(days=random.randint(0,365))).strftime('%Y-%m-%d') for _ in range(200)],
    'value': np.round(np.random.exponential(150, 200), 2)
})
events.to_sql('customer_events', conn, if_exists='replace', index=False)

conn.close()
print('✅ Database seeded at', DB_PATH)
print('   Tables: monthly_revenue, products, customer_events')

---
## Cell 3 · MCP-Style Tool Registry
> Each function below is a **tool** Claude can call. This mirrors MCP's concept of exposing server-side capabilities.

In [ ]:
import sqlite3, json, io, base64
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless for Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LinearRegression
import numpy as np

DB_PATH = '/content/company_data.db'

# ────────────────────────────────────────────────────────────────────────────
# Tool 1 · list_tables  (MCP: schema discovery)
# ────────────────────────────────────────────────────────────────────────────
def list_tables() -> str:
    """Return all tables and their columns from the database."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [row[0] for row in cursor.fetchall()]
    schema = {}
    for t in tables:
        cursor.execute(f'PRAGMA table_info({t})')
        schema[t] = [row[1] for row in cursor.fetchall()]
    conn.close()
    return json.dumps(schema, indent=2)

# ────────────────────────────────────────────────────────────────────────────
# Tool 2 · execute_sql  (MCP: data retrieval)
# ────────────────────────────────────────────────────────────────────────────
def execute_sql(query: str) -> str:
    """Execute a SQL SELECT query and return results as JSON."""
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql_query(query, conn)
        conn.close()
        return df.to_json(orient='records', indent=2)
    except Exception as e:
        return json.dumps({'error': str(e)})

# ────────────────────────────────────────────────────────────────────────────
# Tool 3 · plot_chart  (MCP: visualization)
# ────────────────────────────────────────────────────────────────────────────
def plot_chart(data_json: str, chart_type: str, x_col: str, y_col: str, title: str) -> str:
    """Generate a chart from JSON data. Returns base64 PNG."""
    try:
        df = pd.DataFrame(json.loads(data_json))

        fig, ax = plt.subplots(figsize=(10, 5))
        fig.patch.set_facecolor('#0f1117')
        ax.set_facecolor('#0f1117')

        colors = ['#00d4ff', '#7c3aed', '#f59e0b', '#10b981', '#ef4444']

        if chart_type == 'line':
            ax.plot(df[x_col].astype(str), df[y_col], color=colors[0],
                    linewidth=2.5, marker='o', markersize=5)
            ax.fill_between(range(len(df)), df[y_col], alpha=0.15, color=colors[0])
        elif chart_type == 'bar':
            bars = ax.bar(df[x_col].astype(str), df[y_col], color=colors, edgecolor='none')
        elif chart_type == 'scatter':
            ax.scatter(df[x_col], df[y_col], color=colors[1], alpha=0.7, s=60)
        elif chart_type == 'pie':
            ax.pie(df[y_col], labels=df[x_col], colors=colors,
                   autopct='%1.1f%%', textprops={'color':'white'})

        ax.set_title(title, color='white', fontsize=14, fontweight='bold', pad=15)
        ax.tick_params(colors='#aaaaaa', labelsize=9)
        ax.xaxis.label.set_color('#aaaaaa')
        ax.yaxis.label.set_color('#aaaaaa')
        for spine in ax.spines.values():
            spine.set_edgecolor('#333333')
        ax.set_xlabel(x_col, color='#aaaaaa')
        ax.set_ylabel(y_col, color='#aaaaaa')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()

        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=130, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        plt.close()
        buf.seek(0)
        return base64.b64encode(buf.read()).decode('utf-8')
    except Exception as e:
        return json.dumps({'error': str(e)})

# ────────────────────────────────────────────────────────────────────────────
# Tool 4 · forecast  (MCP: predictive ML)
# ────────────────────────────────────────────────────────────────────────────
def forecast(data_json: str, y_col: str, periods: int = 6) -> str:
    """Run a linear regression forecast and return chart + predictions."""
    try:
        df = pd.DataFrame(json.loads(data_json))
        y = df[y_col].values
        X = np.arange(len(y)).reshape(-1, 1)

        model = LinearRegression().fit(X, y)
        future_X = np.arange(len(y), len(y) + periods).reshape(-1, 1)
        predicted = model.predict(future_X)

        fig, ax = plt.subplots(figsize=(10, 5))
        fig.patch.set_facecolor('#0f1117')
        ax.set_facecolor('#0f1117')

        ax.plot(range(len(y)), y, color='#00d4ff', linewidth=2.5,
                label='Historical', marker='o', markersize=4)
        ax.plot(range(len(y)-1, len(y)+periods), [y[-1]]+list(predicted),
                color='#f59e0b', linewidth=2.5, linestyle='--',
                label=f'{periods}-period Forecast', marker='s', markersize=4)
        ax.axvline(len(y)-1, color='#555', linestyle=':', linewidth=1.5)
        ax.fill_between(range(len(y)-1, len(y)+periods),
                        [y[-1]]+list(predicted * 0.9),
                        [y[-1]]+list(predicted * 1.1),
                        alpha=0.15, color='#f59e0b', label='±10% CI')

        ax.set_title(f'{y_col} · {periods}-Period Forecast', color='white',
                     fontsize=14, fontweight='bold')
        ax.tick_params(colors='#aaaaaa')
        ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
        for spine in ax.spines.values():
            spine.set_edgecolor('#333333')
        plt.tight_layout()

        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=130, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        plt.close()
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode('utf-8')

        return json.dumps({
            'predictions': [round(float(v), 2) for v in predicted],
            'r2_score': round(model.score(X, y), 4),
            'chart_b64': img_b64
        })
    except Exception as e:
        return json.dumps({'error': str(e)})

# ── Tool dispatcher ──────────────────────────────────────────────────────────
TOOL_FUNCTIONS = {
    'list_tables': list_tables,
    'execute_sql': execute_sql,
    'plot_chart': plot_chart,
    'forecast': forecast,
}

print('✅ MCP-style tool registry ready:', list(TOOL_FUNCTIONS.keys()))

---
## Cell 4 · Claude Agent Loop

In [ ]:
import anthropic, json

# ── Tool schemas (sent to Claude so it knows what it can call) ───────────────
TOOLS = [
    {
        'name': 'list_tables',
        'description': 'Browse the database schema. Returns all table names and their columns. Call this FIRST before writing SQL.',
        'input_schema': {'type': 'object', 'properties': {}, 'required': []}
    },
    {
        'name': 'execute_sql',
        'description': 'Execute a SQL SELECT query against the SQLite database and return results as JSON.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'Valid SQLite SELECT query'}
            },
            'required': ['query']
        }
    },
    {
        'name': 'plot_chart',
        'description': 'Generate a visualization chart from JSON data. Returns a base64 PNG image.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'data_json': {'type': 'string', 'description': 'JSON array of records (from execute_sql output)'},
                'chart_type': {'type': 'string', 'enum': ['line', 'bar', 'scatter', 'pie']},
                'x_col': {'type': 'string', 'description': 'Column name for X axis'},
                'y_col': {'type': 'string', 'description': 'Column name for Y axis'},
                'title': {'type': 'string', 'description': 'Chart title'}
            },
            'required': ['data_json', 'chart_type', 'x_col', 'y_col', 'title']
        }
    },
    {
        'name': 'forecast',
        'description': 'Run a machine learning forecast (linear regression) on a numeric column. Returns predictions and a chart.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'data_json': {'type': 'string', 'description': 'JSON array of records (from execute_sql)'},
                'y_col': {'type': 'string', 'description': 'The numeric column to forecast'},
                'periods': {'type': 'integer', 'description': 'Number of future periods to forecast', 'default': 6}
            },
            'required': ['data_json', 'y_col']
        }
    }
]

SYSTEM_PROMPT = """You are Omni, an expert data analyst AI agent.
You have access to a company SQLite database with sales, revenue, and customer data.

ALWAYS follow this workflow:
1. Call list_tables() first to understand the schema.
2. Write and execute precise SQL queries with execute_sql().
3. Visualize results with plot_chart() when the user wants a chart.
4. Use forecast() when users ask about predictions, trends, or future projections.

Be concise, data-driven, and proactive. After showing data, always provide a brief insight.
When using forecast(), mention the R² score to communicate model confidence."""


def run_agent(user_message: str, api_key: str) -> dict:
    """
    Full agentic loop: Claude ↔ tools ↔ Claude
    Returns dict with 'text', 'charts' (list of b64 PNGs), 'tables' (list of DataFrames as HTML)
    """
    client = anthropic.Anthropic(api_key=api_key)
    messages = [{'role': 'user', 'content': user_message}]

    result = {'text': '', 'charts': [], 'tables': []}
    MAX_ITERATIONS = 8  # prevent runaway loops

    for iteration in range(MAX_ITERATIONS):
        response = client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages
        )

        # Collect text from this response
        for block in response.content:
            if block.type == 'text':
                result['text'] += block.text + '\n'

        # If Claude is done, break
        if response.stop_reason == 'end_turn':
            break

        # Process tool calls
        if response.stop_reason == 'tool_use':
            messages.append({'role': 'assistant', 'content': response.content})
            tool_results = []

            for block in response.content:
                if block.type != 'tool_use':
                    continue

                tool_name = block.name
                tool_input = block.input
                tool_id = block.id

                print(f'  🔧 Calling tool: {tool_name}({list(tool_input.keys())})')

                # Dispatch to the right function
                fn = TOOL_FUNCTIONS.get(tool_name)
                if fn:
                    tool_output = fn(**tool_input)
                else:
                    tool_output = json.dumps({'error': f'Unknown tool: {tool_name}'})

                # Intercept chart/forecast outputs for the UI
                if tool_name == 'plot_chart':
                    result['charts'].append(tool_output)  # raw b64
                    tool_output = json.dumps({'status': 'chart_generated', 'message': 'Chart created successfully.'})

                elif tool_name == 'forecast':
                    parsed = json.loads(tool_output)
                    if 'chart_b64' in parsed:
                        result['charts'].append(parsed.pop('chart_b64'))
                    tool_output = json.dumps(parsed)

                elif tool_name == 'execute_sql':
                    try:
                        df = pd.read_json(tool_output)
                        result['tables'].append(df.to_html(classes='data-table', index=False))
                    except:
                        pass

                tool_results.append({
                    'type': 'tool_result',
                    'tool_use_id': tool_id,
                    'content': tool_output
                })

            messages.append({'role': 'user', 'content': tool_results})

    return result


print('✅ Claude agent loop ready')

---
## Cell 5 · Streamlit App
> Writes `app.py` to disk. Run Cell 6 to launch it.

In [ ]:
APP_CODE = '''
import streamlit as st
import base64, sys, os
from io import BytesIO
from PIL import Image
import pandas as pd

sys.path.insert(0, "/content")

# ── Page config ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Omni Data Analyst",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Custom CSS ───────────────────────────────────────────────────────────────
st.markdown("""
<style>
@import url(\'https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Inter:wght@300;400;600&display=swap\');

html, body, [class*="css"] { font-family: \'Inter\', sans-serif; }

.stApp { background: #0a0a12; color: #e0e0e0; }

.hero-title {
    font-family: \'Space Mono\', monospace;
    font-size: 2.2rem;
    font-weight: 700;
    background: linear-gradient(135deg, #00d4ff 0%, #7c3aed 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin-bottom: 0.2rem;
}
.hero-sub { color: #888; font-size: 0.9rem; margin-bottom: 2rem; }

.chip {
    display: inline-block;
    background: rgba(0,212,255,0.1);
    border: 1px solid rgba(0,212,255,0.3);
    border-radius: 20px;
    padding: 3px 12px;
    font-size: 0.75rem;
    color: #00d4ff;
    margin: 2px;
}

.answer-box {
    background: #111827;
    border-left: 3px solid #00d4ff;
    border-radius: 8px;
    padding: 1.2rem 1.5rem;
    margin: 1rem 0;
    line-height: 1.7;
    white-space: pre-wrap;
}

.data-table {
    font-size: 0.82rem !important;
    border-collapse: collapse;
    width: 100%;
}
.data-table th {
    background: rgba(124,58,237,0.3);
    color: #c4b5fd;
    padding: 6px 12px;
    text-align: left;
    border-bottom: 1px solid #333;
}
.data-table td {
    padding: 5px 12px;
    border-bottom: 1px solid #1f2937;
    color: #d1d5db;
}
.data-table tr:hover td { background: rgba(255,255,255,0.03); }

.metric-card {
    background: linear-gradient(135deg, #111827 0%, #1a1a2e 100%);
    border: 1px solid #1f2937;
    border-radius: 10px;
    padding: 1rem;
    text-align: center;
}
.metric-value { font-size: 1.6rem; font-weight: 700; color: #00d4ff; }
.metric-label { font-size: 0.75rem; color: #6b7280; margin-top: 4px; }

div[data-testid="stSidebar"] { background: #080810; border-right: 1px solid #1f2937; }
div[data-testid="stSidebar"] * { color: #d1d5db !important; }

textarea { background: #111827 !important; color: #e0e0e0 !important; border: 1px solid #374151 !important; border-radius: 8px !important; }
button[kind="primary"] { background: linear-gradient(135deg, #7c3aed, #00d4ff) !important; border: none !important; }
</style>
""", unsafe_allow_html=True)

# ── Sidebar ──────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown(\'### 🤖 Omni Data Analyst\')
    st.markdown(\'---\')
    api_key = st.text_input(\'Anthropic API Key\', type=\'password\', placeholder=\'sk-ant-...\')
    st.markdown(\'---\')
    st.markdown(\'**🛠 Available Tools**\')
    for tool, desc in [
        (\'🗄 list_tables\', \'Schema discovery\'),
        (\'⚡ execute_sql\', \'Run SQL queries\'),
        (\'📊 plot_chart\', \'Visualize data\'),
        (\'🔮 forecast\', \'ML predictions\'),
    ]:
        st.markdown(f\' **{tool}** — {desc}\')
    st.markdown(\'---\')
    st.markdown(\'**💡 Try asking:**\')
    examples = [
        \'Show me monthly revenue growth as a line chart\',
        \'Which product category has the highest sales?\',
        \'Forecast revenue for the next 6 months\',
        \'What is the customer churn rate this year?\',
        \'Show top 5 months by new customers with a bar chart\',
    ]
    for ex in examples:
        if st.button(ex, use_container_width=True):
            st.session_state[\'prefill\'] = ex

# ── Main area ────────────────────────────────────────────────────────────────
st.markdown(\'<div class="hero-title">⬡ Omni Data Analyst</div>\', unsafe_allow_html=True)
st.markdown(\'<div class="hero-sub">Natural language → SQL → Visualization → Insights</div>\', unsafe_allow_html=True)

for chip in [\'Claude Tool Use\', \'MCP Architecture\', \'Scikit-learn\', \'SQLite\', \'Matplotlib\']:
    st.markdown(f\'<span class="chip">{chip}</span>\', unsafe_allow_html=True)

st.markdown(\'<br>\', unsafe_allow_html=True)

# Input
default_val = st.session_state.pop(\'prefill\', \'\')
query = st.text_area(\'Ask anything about your data\', value=default_val, height=100,
                     placeholder=\'e.g. Show me monthly revenue growth with a line chart...\')

run_btn = st.button(\'🚀 Analyze\', type=\'primary\', use_container_width=True)

# Run agent
if run_btn:
    if not api_key:
        st.error(\'⚠️ Please enter your Anthropic API key in the sidebar.\')
    elif not query.strip():
        st.warning(\'Please enter a question.\')
    else:
        from agent import run_agent  # import from the agent module we wrote
        with st.spinner(\'🤖 Omni is thinking and querying your database...\'): 
            result = run_agent(query, api_key)

        st.markdown(\'### 📋 Analysis\')
        if result[\'text\'].strip():
            st.markdown(f\'<div class="answer-box">{result["text"]}</div>\', unsafe_allow_html=True)

        if result[\'charts\']:
            st.markdown(\'### 📊 Visualizations\')
            cols = st.columns(min(len(result[\'charts\']), 2))
            for i, chart_b64 in enumerate(result[\'charts\']):
                img_bytes = base64.b64decode(chart_b64)
                img = Image.open(BytesIO(img_bytes))
                cols[i % 2].image(img, use_column_width=True)

        if result[\'tables\']:
            st.markdown(\'### 🗃 Query Results\')
            for tbl_html in result[\'tables\']:
                st.markdown(tbl_html, unsafe_allow_html=True)
                st.markdown(\'<br>\', unsafe_allow_html=True)
'''

with open('/content/app.py', 'w') as f:
    f.write(APP_CODE)

# Also write the agent module so Streamlit can import it
import inspect

AGENT_MODULE = '''
import sqlite3, json, io, base64
import pandas as pd
import matplotlib
matplotlib.use(\'Agg\')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import numpy as np
import anthropic

DB_PATH = \'/content/company_data.db\'

def list_tables():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type=\'table\'")
    tables = [row[0] for row in cursor.fetchall()]
    schema = {}
    for t in tables:
        cursor.execute(f\'PRAGMA table_info({t})\')
        schema[t] = [row[1] for row in cursor.fetchall()]
    conn.close()
    return json.dumps(schema, indent=2)

def execute_sql(query):
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql_query(query, conn)
        conn.close()
        return df.to_json(orient=\'records\', indent=2)
    except Exception as e:
        return json.dumps({\'error\': str(e)})

def plot_chart(data_json, chart_type, x_col, y_col, title):
    try:
        df = pd.DataFrame(json.loads(data_json))
        fig, ax = plt.subplots(figsize=(10, 5))
        fig.patch.set_facecolor(\'#0f1117\')
        ax.set_facecolor(\'#0f1117\')
        colors = [\'#00d4ff\', \'#7c3aed\', \'#f59e0b\', \'#10b981\', \'#ef4444\']
        if chart_type == \'line\':
            ax.plot(df[x_col].astype(str), df[y_col], color=colors[0], linewidth=2.5, marker=\'o\', markersize=5)
            ax.fill_between(range(len(df)), df[y_col], alpha=0.15, color=colors[0])
        elif chart_type == \'bar\':
            ax.bar(df[x_col].astype(str), df[y_col], color=colors, edgecolor=\'none\')
        elif chart_type == \'scatter\':
            ax.scatter(df[x_col], df[y_col], color=colors[1], alpha=0.7, s=60)
        elif chart_type == \'pie\':
            ax.pie(df[y_col], labels=df[x_col], colors=colors, autopct=\'%1.1f%%\', textprops={\'color\':\'white\'})
        ax.set_title(title, color=\'white\', fontsize=14, fontweight=\'bold\', pad=15)
        ax.tick_params(colors=\'#aaaaaa\', labelsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor(\'#333333\')
        ax.set_xlabel(x_col, color=\'#aaaaaa\')
        ax.set_ylabel(y_col, color=\'#aaaaaa\')
        plt.xticks(rotation=45, ha=\'right\')
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format=\'png\', dpi=130, bbox_inches=\'tight\', facecolor=fig.get_facecolor())
        plt.close()
        buf.seek(0)
        return base64.b64encode(buf.read()).decode(\'utf-8\')
    except Exception as e:
        return json.dumps({\'error\': str(e)})

def forecast(data_json, y_col, periods=6):
    try:
        df = pd.DataFrame(json.loads(data_json))
        y = df[y_col].values
        X = np.arange(len(y)).reshape(-1, 1)
        model = LinearRegression().fit(X, y)
        future_X = np.arange(len(y), len(y)+periods).reshape(-1, 1)
        predicted = model.predict(future_X)
        fig, ax = plt.subplots(figsize=(10, 5))
        fig.patch.set_facecolor(\'#0f1117\')
        ax.set_facecolor(\'#0f1117\')
        ax.plot(range(len(y)), y, color=\'#00d4ff\', linewidth=2.5, label=\'Historical\', marker=\'o\', markersize=4)
        ax.plot(range(len(y)-1, len(y)+periods), [y[-1]]+list(predicted), color=\'#f59e0b\', linewidth=2.5, linestyle=\'--\', label=f\'{periods}-period Forecast\', marker=\'s\', markersize=4)
        ax.fill_between(range(len(y)-1, len(y)+periods), [y[-1]]+list(predicted*0.9), [y[-1]]+list(predicted*1.1), alpha=0.15, color=\'#f59e0b\')
        ax.set_title(f\'{y_col} · {periods}-Period Forecast\', color=\'white\', fontsize=14, fontweight=\'bold\')
        ax.tick_params(colors=\'#aaaaaa\')
        ax.legend(facecolor=\'#1a1a2e\', labelcolor=\'white\', fontsize=9)
        for spine in ax.spines.values(): spine.set_edgecolor(\'#333333\')
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format=\'png\', dpi=130, bbox_inches=\'tight\', facecolor=fig.get_facecolor())
        plt.close()
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode(\'utf-8\')
        return json.dumps({\'predictions\': [round(float(v),2) for v in predicted], \'r2_score\': round(model.score(X,y),4), \'chart_b64\': img_b64})
    except Exception as e:
        return json.dumps({\'error\': str(e)})

TOOL_FUNCTIONS = {\'list_tables\': list_tables, \'execute_sql\': execute_sql, \'plot_chart\': plot_chart, \'forecast\': forecast}

TOOLS = [
    {\'name\': \'list_tables\', \'description\': \'Browse database schema. Call this first before writing SQL.\', \'input_schema\': {\'type\': \'object\', \'properties\': {}, \'required\': []}},
    {\'name\': \'execute_sql\', \'description\': \'Execute a SQL SELECT query.\', \'input_schema\': {\'type\': \'object\', \'properties\': {\'query\': {\'type\': \'string\'}}, \'required\': [\'query\']}},
    {\'name\': \'plot_chart\', \'description\': \'Generate a visualization chart.\', \'input_schema\': {\'type\': \'object\', \'properties\': {\'data_json\': {\'type\': \'string\'}, \'chart_type\': {\'type\': \'string\', \'enum\': [\'line\',\'bar\',\'scatter\',\'pie\']}, \'x_col\': {\'type\': \'string\'}, \'y_col\': {\'type\': \'string\'}, \'title\': {\'type\': \'string\'}}, \'required\': [\'data_json\',\'chart_type\',\'x_col\',\'y_col\',\'title\']}},
    {\'name\': \'forecast\', \'description\': \'ML forecast on a numeric column.\', \'input_schema\': {\'type\': \'object\', \'properties\': {\'data_json\': {\'type\': \'string\'}, \'y_col\': {\'type\': \'string\'}, \'periods\': {\'type\': \'integer\'}}, \'required\': [\'data_json\',\'y_col\']}},
]

SYSTEM_PROMPT = """You are Omni, an expert data analyst AI. You have access to a company SQLite database.
ALWAYS: 1) Call list_tables() first. 2) Write precise SQL. 3) Visualize with plot_chart(). 4) Use forecast() for predictions.
Be concise and data-driven. Mention R² score when forecasting."""

def run_agent(user_message, api_key):
    client = anthropic.Anthropic(api_key=api_key)
    messages = [{\'role\': \'user\', \'content\': user_message}]
    result = {\'text\': \'\', \'charts\': [], \'tables\': []}
    for _ in range(8):
        response = client.messages.create(model=\'claude-sonnet-4-20250514\', max_tokens=4096, system=SYSTEM_PROMPT, tools=TOOLS, messages=messages)
        for block in response.content:
            if block.type == \'text\': result[\'text\'] += block.text + \'\\n\'
        if response.stop_reason == \'end_turn\': break
        if response.stop_reason == \'tool_use\':
            messages.append({\'role\': \'assistant\', \'content\': response.content})
            tool_results = []
            for block in response.content:
                if block.type != \'tool_use\': continue
                fn = TOOL_FUNCTIONS.get(block.name)
                tool_output = fn(**block.input) if fn else json.dumps({\'error\': f\'Unknown tool: {block.name}\'})
                if block.name == \'plot_chart\':
                    result[\'charts\'].append(tool_output)
                    tool_output = json.dumps({\'status\': \'chart_generated\'})
                elif block.name == \'forecast\':
                    parsed = json.loads(tool_output)
                    if \'chart_b64\' in parsed: result[\'charts\'].append(parsed.pop(\'chart_b64\'))
                    tool_output = json.dumps(parsed)
                elif block.name == \'execute_sql\':
                    try:
                        df = pd.read_json(tool_output)
                        result[\'tables\'].append(df.to_html(classes=\'data-table\', index=False))
                    except: pass
                tool_results.append({\'type\': \'tool_result\', \'tool_use_id\': block.id, \'content\': tool_output})
            messages.append({\'role\': \'user\', \'content\': tool_results})
    return result
'''

with open('/content/agent.py', 'w') as f:
    f.write(AGENT_MODULE)

print('✅ app.py and agent.py written to /content/')

---
## Cell 6 · 🚀 Launch the App
> This starts Streamlit and tunnels it through ngrok so you get a public URL.

In [ ]:
# ── Get your free ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ''  # ← paste your token here (free account works)

# ────────────────────────────────────────────────────────────────────────────
import subprocess, time, threading
from pyngrok import ngrok, conf

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

# Kill any existing Streamlit
!pkill -f streamlit 2>/dev/null; sleep 1

# Start Streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', '/content/app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print(f'\n🌐 App is LIVE at: {public_url}')
print('   Open the link above in any browser!')
print('   Enter your Anthropic API key in the sidebar.')
print('\n⚠️  To stop: run  ngrok.disconnect(public_url)  or restart runtime')

---
## (Optional) Cell 7 · Quick CLI Test — No UI Needed
> Test the agent directly in the notebook without launching the browser UI.

In [ ]:
import base64
from IPython.display import display, Image as IPImage, HTML

API_KEY = ''  # ← paste your Anthropic API key here
TEST_QUERY = 'Show me the monthly revenue trend as a line chart and forecast the next 4 months.'

if API_KEY:
    print(f'🔍 Query: {TEST_QUERY}\n')
    result = run_agent(TEST_QUERY, API_KEY)

    print('\n📋 Analysis:\n', result['text'])

    for i, chart_b64 in enumerate(result['charts']):
        print(f'\n📊 Chart {i+1}:')
        display(IPImage(data=base64.b64decode(chart_b64)))

    if result['tables']:
        print('\n🗃 Data Table:')
        display(HTML(result['tables'][0]))
else:
    print('⚠️  Set API_KEY above to run the test')

---
## 📐 Architecture Reference

| Layer | Technology | Role |
|---|---|---|
| **Brain** | Claude Sonnet via Anthropic API | Reasoning, SQL generation, tool orchestration |
| **Bridge** | Tool Use / MCP-style registry | Exposes database & ML as callable functions |
| **Database** | SQLite + Pandas | Data storage and retrieval |
| **Visualization** | Matplotlib | Chart generation |
| **Forecasting** | Scikit-learn LinearRegression | Predictive analytics |
| **Frontend** | Streamlit | Interactive web UI |
| **Tunnel** | ngrok | Expose Colab to the internet |

### How to use your own database
1. Upload your `.db` or `.sqlite` file to Colab
2. Change `DB_PATH` in Cell 3 to point to your file
3. Re-run Cells 3–6

### How to add a new tool
1. Write a Python function in Cell 3
2. Add it to `TOOL_FUNCTIONS` dict
3. Add its JSON schema to `TOOLS` list in Cell 4
4. Claude will automatically discover and use it!
